In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
imdb_id = "tt0468569"
url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

In [3]:
response.text[:400]

'<!doctype html><html class="a-no-js" data-19ax5a9jf="dingo"><head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var '

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
mpaa_label = soup.find(string="MPAA")
mpaa_rating = mpaa_label.find_parent().find_next_sibling().get_text(strip=True)
mpaa_rating

'PG-13'

In [6]:
dist_label = soup.find(string=lambda s: s and "Distributor" in s)
distributor = dist_label.find_parent().find_next_sibling().get_text(strip=True).split("See full")[0].strip()
distributor

'Warner Bros.'

In [7]:
movie_data = {"imdb_id": imdb_id, "mpaa_rating": mpaa_rating, "distributor": distributor}
movie_data

{'imdb_id': 'tt0468569', 'mpaa_rating': 'PG-13', 'distributor': 'Warner Bros.'}

In [8]:
import time

df = pd.read_csv("data/processed/tmdb_movies.csv")
imdb_ids = df["imdb_id"].dropna()
movies = []

for imdb_id in imdb_ids:
    url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    try:
        mpaa_rating = soup.find(string="MPAA").find_parent().find_next_sibling().get_text(strip=True)
    except AttributeError:
        mpaa_rating = None

    try:
        distributor = soup.find(string=lambda s: s and "Distributor" in s).find_parent().find_next_sibling().get_text(strip=True).split("See full")[0].strip()
    except AttributeError:
        distributor = None

    movies.append({"imdb_id": imdb_id, "mpaa_rating": mpaa_rating, "distributor": distributor})
    time.sleep(1)

In [9]:
pd.DataFrame(movies).to_csv("boxofficemojo_predictors.csv", index=False)